# Lab 16: two GPUs

Kaggle gives you two T4s. One GPU, then tensor parallel across both, then two data-parallel copies: the step, the shelf and the throughput for each.

In [ ]:
# Cell 1: the GPU this notebook got. On Kaggle: Settings → Accelerator → GPU T4 x2, then Run all.
import subprocess, sys
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True)
if out.returncode != 0 or not out.stdout.strip():
    sys.exit("No GPU. Open Settings → Accelerator and choose GPU T4 x2, then run this cell again.")
print(out.stdout.strip())
if "T4" in out.stdout:
    print("Path: Kaggle T4. Labs 08 and 10 are read-only on this GPU (no FP8 arithmetic).")

In [ ]:
# Cell 2: install the pinned engine. About nine minutes on Kaggle. Kaggle's preinstalled torchaudio is built
# for an older CUDA and breaks the import, so it is removed.
%pip install -q "vllm==0.32.0" 2>&1 | tail -2
%pip uninstall -y -q torchaudio
import vllm, torch
print("vllm", vllm.__version__, "· torch", torch.__version__, "· compute capability", torch.cuda.get_device_capability())

In [ ]:
# Cell 3: helpers used by every lab: start the server in the background, wait for it, talk to it.
import json, re, subprocess, time, urllib.request
MODEL = "Qwen/Qwen3-4B-Instruct-2507"
URL = "http://127.0.0.1:8000"

def serve(extra_args=(), log_path="/kaggle/working/serve.log", timeout_s=900):
    args = ["vllm", "serve", MODEL, "--port", "8000", "--dtype", "half", *extra_args]   # the T4 has no bfloat16
    log = open(log_path, "w")
    t0 = time.time()
    proc = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT)
    while time.time() - t0 < timeout_s:
        if proc.poll() is not None:
            raise RuntimeError("vllm serve exited; see " + log_path)
        try:
            urllib.request.urlopen(URL + "/health", timeout=2); return proc, round(time.time() - t0, 1)
        except Exception:
            time.sleep(0.5)
    raise RuntimeError("server not ready in time")

def stop(proc):
    proc.terminate(); proc.wait()

def log_lines(path, patterns):
    text = open(path).read(); out = {}
    for name, pat in patterns.items():
        m = re.search(pat, text)
        if m:
            line = re.sub(r"^\(EngineCore[^)]*\) ", "", m.group(0)); line = re.sub(r"^INFO \d\d-\d\d \d\d:\d\d:\d\d \[[^\]]+\] ", "", line); out[name] = line.strip()
    return out

STARTUP = {"dtype": r"INFO[^\n]*dtype=torch\.[a-z0-9]+", "backend": r"Using [A-Z_]+ attention backend[^\n]*", "weights": r"Loading weights took [^\n]*",
           "model": r"Model loading took [^\n]*", "kv_memory": r"Available KV cache memory: [^\n]*", "kv_size": r"GPU KV cache size: [^\n]*", "graphs": r"Graph capturing finished[^\n]*"}

def chat(messages, **params):
    body = json.dumps({"model": MODEL, "messages": messages, **params}).encode()
    req = urllib.request.Request(URL + "/v1/chat/completions", data=body, headers={"Content-Type": "application/json"})
    return json.loads(urllib.request.urlopen(req, timeout=300).read())
print("helpers ready")

In [ ]:
# Cell 4: the benchmark call and the measurement.
def bench_quiet(rate, num_prompts, input_len=1024, output_len=128, max_concurrency=None, model=None):
    args = ["vllm", "bench", "serve", "--model", model or MODEL, "--tokenizer", MODEL, "--port", "8000", "--dataset-name", "random", "--random-input-len", str(input_len),
            "--random-output-len", str(output_len), "--num-prompts", str(num_prompts), "--request-rate", str(rate),
            "--metric-percentiles", "50,95,99", "--percentile-metrics", "ttft,tpot,itl,e2el", "--ignore-eos", "--seed", "0"]
    if max_concurrency: args += ["--max-concurrency", str(max_concurrency)]
    out = subprocess.run(args, capture_output=True, text=True); text = out.stdout; i = text.find("Serving Benchmark Result")
    res = {}
    for line in text.splitlines():
        m = re.match(r'^(.*?):\s+([0-9.]+)\s*$', line)
        if m: res[m.group(1).strip()] = float(m.group(2))
    return res
def row(label, r):
    g = lambda k: r.get(k, float("nan"))
    return f"{label:<28} req/s {g('Request throughput (req/s)'):5.2f}  out tok/s {g('Output token throughput (tok/s)'):7.1f}  TTFT p50 {g('P50 TTFT (ms)'):6.0f} p99 {g('P99 TTFT (ms)'):6.0f} ms  TPOT p50 {g('P50 TPOT (ms)'):5.1f} p99 {g('P99 TPOT (ms)'):5.1f} ms  peak conc {g('Maximum request concurrency'):3.0f}"

def measure(label):
    for k, v in log_lines("/kaggle/working/serve.log", {"kv": r"GPU KV cache size: [^\n]*"}).items(): print("  " + v)
    print("  " + row(f"{label}: 1 at a time", bench_quiet(rate="inf", num_prompts=16, input_len=512, max_concurrency=1)))
    for rate in (2, 4, 8):
        print("  " + row(f"{label}: {rate} req/s", bench_quiet(rate=rate, num_prompts=64, input_len=512)))

In [ ]:
# Cell 5: one T4.
proc, ready_s = serve(["--max-model-len", "4096"]); print(f"ready after {ready_s} s"); measure("1 GPU"); stop(proc)

In [ ]:
# Cell 6: tensor parallel across both.
proc, ready_s = serve(["--max-model-len", "4096", "--tensor-parallel-size", "2"]); print(f"ready after {ready_s} s"); measure("TP=2"); stop(proc)

In [ ]:
# Cell 7: two data-parallel copies.
proc, ready_s = serve(["--max-model-len", "4096", "--data-parallel-size", "2"]); print(f"ready after {ready_s} s"); measure("DP=2"); stop(proc)